In [1]:
!pip install -q open-clip-torch sentence-transformers langchain-text-splitters tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import torch
import traceback
import numpy as np
import pandas as pd
import open_clip
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & HẰNG SỐ
# ==========================================
INPUT_JSON = '/kaggle/input/datasets/baubauuu/vn1232131/vietnam_only.json'
OUTPUT_DIR = '/kaggle/working/Milvus_Export_Data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_SUM_PQ     = os.path.join(OUTPUT_DIR, 'milvus_summary.parquet')
OUT_CHK_PQ     = os.path.join(OUTPUT_DIR, 'milvus_chunks.parquet')
PROGRESS_FILE  = os.path.join(OUTPUT_DIR, 'processed_ids.txt')
ERROR_LOG_FILE = os.path.join(OUTPUT_DIR, 'error_log.txt')

DIM_CLIP = 1024 # Khớp với ViT-H-14
DIM_E5   = 1024 # Khớp với multilingual-e5-large
BATCH_SIZE = 100

In [ ]:
# 2. KHỞI TẠO MODEL & DEVICE
# ==========================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f" Thiết bị đang sử dụng: {device.upper()}")

# Load CLIP
print(" Đang nạp Model CLIP (ViT-H-14)...")
model_clip, _, _ = open_clip.create_model_and_transforms('ViT-H-14', pretrained='laion2b_s32b_b79k')
tokenizer_clip = open_clip.get_tokenizer('ViT-H-14')
model_clip = model_clip.to(device).eval()

# Load E5
print(" Đang nạp Model E5 (Multilingual-Large)...")
model_text = SentenceTransformer('intfloat/multilingual-e5-large', device=device)

🚀 Thiết bị đang sử dụng: CUDA
⏳ Đang nạp Model CLIP (ViT-H-14)...


open_clip_model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

⏳ Đang nạp Model E5 (Multilingual-Large)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [4]:
# ==========================================
# 3. CÁC HÀM HỖ TRỢ
# ==========================================
def load_processed_ids():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def is_valid_vec(vec, expected_dim):
    if vec is None: return False
    v = np.array(vec)
    if v.size == 0 or v.shape[-1] != expected_dim: return False
    if np.isnan(v).any() or np.isinf(v).any(): return False
    return True

def clean_txt(text):
    if not text: return ""
    text = re.sub(r'\[\d+\]|\[citation needed\]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def enrich_and_split(item, splitter):
    title = item.get("title", "Unknown")
    full_text = item.get("full_text", "")
    if not full_text or len(full_text.strip()) < 50: return []

    parts = re.split(r'(==+.*?==+)', full_text)
    current_header = "General Information"
    final_chunks = []

    for part in parts:
        part = part.strip()
        if not part: continue
        if re.match(r'==+.*?==+', part):
            current_header = part.strip('= ').strip()
            continue
        
        blacklist = ['references', 'see also', 'external links', 'notes']
        if current_header.lower() in blacklist: continue

        clean_part = clean_txt(part)
        if len(clean_part) < 20: continue

        sub_chunks = splitter.split_text(clean_part)
        for c in sub_chunks:
            # Gắn context cho chunk để search chính xác hơn
            enriched = f"[{title} - {current_header}] {c}"
            final_chunks.append(enriched)
    return final_chunks

def save_append_parquet(new_data, file_path):
    if not new_data: return
    df_new = pd.DataFrame(new_data)
    if os.path.exists(file_path):
        df_old = pd.read_parquet(file_path)
        pd.concat([df_old, df_new], ignore_index=True).to_parquet(file_path, index=False)
    else:
        df_new.to_parquet(file_path, index=False)

In [ ]:
# ==========================================
# 4. QUY TRÌNH XỬ LÝ CHÍNH
# ==========================================
def process_and_export():
    splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
    
    print(f" Đang mở file {INPUT_JSON}...")
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)

    processed_ids = load_processed_ids()
    items_to_process = [item for item in data if str(item.get("id_dia_diem")) not in processed_ids]
    
    if not items_to_process:
        print(" Tuyệt vời! Dữ liệu đã được xử lý hoàn tất từ trước.")
        return

    print(f" Bắt đầu xử lý {len(items_to_process)} địa điểm mới...")
    
    batch_summaries = []
    batch_chunks = []
    count_summary, count_chunk = 0, 0

    pbar = tqdm(items_to_process, desc="Mining Wikipedia")

    for i, item in enumerate(pbar):
        idx = str(item.get("id_dia_diem"))
        title = item.get("title", "Unknown")

        try:
            # --- 1. CLIP Summary ---
            s_txt = f"[{title}] {item.get('summary_text', '')}"
            with torch.no_grad():
                tokens = tokenizer_clip([clean_txt(s_txt)]).to(device)
                s_vec = model_clip.encode_text(tokens).cpu().numpy()[0]
                s_vec /= (np.linalg.norm(s_vec) + 1e-10)

            if is_valid_vec(s_vec, DIM_CLIP):
                batch_summaries.append({
                    "id": int(idx), "vector": s_vec.tolist(), "title": title
                })
                count_summary += 1

            # --- 2. E5 Chunks ---
            chunks = enrich_and_split(item, splitter)
            if chunks:
                texts_e5 = ["passage: " + clean_txt(c) for c in chunks]
                # Xử lý batch nhỏ bên trong để tránh nổ VRAM
                c_vecs = model_text.encode(texts_e5, batch_size=32, show_progress_bar=False)

                for j, (txt, v) in enumerate(zip(chunks, c_vecs)):
                    if is_valid_vec(v, DIM_E5):
                        batch_chunks.append({
                            "chunk_id": f"{idx}_{j}",
                            "parent_id": int(idx),
                            "vector": v.tolist(),
                            "content": txt
                        })
                        count_chunk += 1

            pbar.set_postfix({"Sum": count_summary, "Chunks": count_chunk})

            # --- 3. Lưu dữ liệu theo đợt ---
            if (len(batch_summaries) >= BATCH_SIZE) or (i == len(items_to_process) - 1):
                pbar.set_description("💾 Saving Parquet")
                
                save_append_parquet(batch_summaries, OUT_SUM_PQ)
                save_append_parquet(batch_chunks, OUT_CHK_PQ)
                
        # THÊM DÒNG NÀY ĐỂ LOG TRONG SAVE VERSION
                print(f" Đã chốt hạ {len(batch_summaries)} địa điểm vào file. Tổng cộng Sum: {count_summary}")
                # Cập nhật checkpoint
                with open(PROGRESS_FILE, 'a') as f_p:
                    for s_item in batch_summaries:
                        f_p.write(str(s_item['id']) + "\n")
                
                batch_summaries, batch_chunks = [], []
                pbar.set_description("Mining Wikipedia")

        except Exception as e:
            with open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f_err:
                f_err.write(f"Lỗi tại ID {idx} ({title}): {str(e)}\n{traceback.format_exc()}\n")
            continue

    print(f"\n" + "="*40)
    print(f" HOÀN TẤT QUY TRÌNH!")
    print(f" Summary vectors: {count_summary}")
    print(f" Chunk vectors:   {count_chunk}")
    print(f" Kết quả tại: {OUTPUT_DIR}")
    print("="*40)

# Chạy trực tiếp
process_and_export()

📂 Đang mở file /kaggle/input/datasets/baubauuu/vn1232131/vietnam_only.json...
🚀 Bắt đầu xử lý 574 địa điểm mới...


Mining Wikipedia:   0%|          | 0/574 [00:00<?, ?it/s]

✅ Đã chốt hạ 100 địa điểm vào file. Tổng cộng Sum: 100
✅ Đã chốt hạ 100 địa điểm vào file. Tổng cộng Sum: 200
✅ Đã chốt hạ 100 địa điểm vào file. Tổng cộng Sum: 300
✅ Đã chốt hạ 100 địa điểm vào file. Tổng cộng Sum: 400
✅ Đã chốt hạ 100 địa điểm vào file. Tổng cộng Sum: 500
✅ Đã chốt hạ 74 địa điểm vào file. Tổng cộng Sum: 574

🏁 HOÀN TẤT QUY TRÌNH!
📊 Summary vectors: 574
📊 Chunk vectors:   2364
💾 Kết quả tại: /kaggle/working/Milvus_Export_Data
